# FinBERT Sentiment Analysis — Headline Bias Experiment

This notebook applies FinBERT — a BERT-based language model fine-tuned on financial 
text (Huang et al., 2023) — to extract sentiment scores from WTI-related news articles. 

Sentiment is extracted independently from two input configurations:

- **Title only:** The article headline as retrieved from GDELT.
- **Title + body:** The headline concatenated with the first three paragraphs of the 
  article body, scraped from the original URL.

This dual extraction enables a direct comparison of sentiment signals derived from 
headlines alone versus full-text content — an experiment designed to quantify 
**headline bias** in financial sentiment analysis. The hypothesis is that headlines, 
written to capture attention rather than to be precise, may systematically diverge 
from the sentiment expressed in the article body, and that this divergence has 
measurable consequences for downstream liquidity modeling.

For each article, FinBERT produces:
- A sentiment label: `positive`, `neutral`, or `negative`
- A confidence score between 0 and 1

Three comparisons are produced:
1. **Divergence rate** — percentage of articles where title and full-text sentiment labels disagree
2. **Score delta** — magnitude of confidence score change between the two input configurations  
3. **Predictive validity** — which configuration better predicts contemporaneous log volume

The resulting sentiment features are merged with `gdelt_wti_aligned.csv` to produce 
the final modeling dataset for the asymmetry (RQ1) and temporal lag (RQ2) analyses.

Output saved to `01_data/features/gdelt_wti_sentiment.csv`.

### General imports

In [11]:
import torch
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset

In [12]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

Device: mps


### Load data

In [22]:
df = pd.read_csv("../01_data/processed/gdelt_wti_aligned.csv", parse_dates=['datetime'])
print(f"Articles loaded: {len(df)}")
print(f"Articles with valid body: {df['body_valid'].sum()}")
print(f"Articles title-only fallback: {(df['body_valid'] != True).sum()}")

# Check articles with missing titles
df_no_title = df[df['title'].isna()]
print(f"Articles with missing title: {len(df_no_title)}")
print(f"Of those, with valid body: {df_no_title['body_valid'].sum()}")

# Discard articles with missing titles
before = len(df)
df = df.dropna(subset=['title']).reset_index(drop=True)
print(f"Dropped {before - len(df)} articles with missing titles")
print(f"Remaining: {len(df)}")

Articles loaded: 13691
Articles with valid body: 7756
Articles title-only fallback: 5935
Articles with missing title: 1
Of those, with valid body: 1
Dropped 1 articles with missing titles
Remaining: 13690


### Load FinBERT

Using HuggingFace - ProsusAI/finbert

In [14]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv("../.env")
login(token=os.environ.get("HUGGINGFACE_TOKEN"))

In [15]:
model_name = "ProsusAI/finbert"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)
model = model.to(device)
model.eval()
print("FinBERT loaded.")
print(f"Labels: {model.config.id2label}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 19924.26it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT loaded.
Labels: {0: 'positive', 1: 'negative', 2: 'neutral'}


### Inference

The function get a text array, executes finBERT with the text and saves the results in a dataframe.

In [16]:
def get_sentiment(texts, batch_size=32):
    results = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
        labels = np.argmax(probs, axis=1)
        
        for label, prob in zip(labels, probs):
            results.append({
                "sentiment": model.config.id2label[label],
                "score_positive": round(float(prob[0]), 4),
                "score_negative": round(float(prob[1]), 4),
                "score_neutral": round(float(prob[2]), 4),
                "confidence": round(float(prob.max()), 4)
            })
        
        if (i + batch_size) % 100 == 0:
            print(f"Processed {min(i+batch_size, len(texts))}/{len(texts)}")
    
    return pd.DataFrame(results)

### Inference on news headers only

This experiments will use only news headers to get a sentiment score.

In [23]:
print("Running FinBERT on titles...")
titles = df['title'].fillna('').tolist()
sentiment_titles = get_sentiment(titles)
sentiment_titles.columns = [f"title_{c}" for c in sentiment_titles.columns]
print(sentiment_titles['title_sentiment'].value_counts())

Running FinBERT on titles...
Processed 800/13690
Processed 1600/13690
Processed 2400/13690
Processed 3200/13690
Processed 4000/13690
Processed 4800/13690
Processed 5600/13690
Processed 6400/13690
Processed 7200/13690
Processed 8000/13690
Processed 8800/13690
Processed 9600/13690
Processed 10400/13690
Processed 11200/13690
Processed 12000/13690
Processed 12800/13690
Processed 13600/13690
title_sentiment
negative    5194
neutral     4522
positive    3974
Name: count, dtype: int64


### FinBERT Sentiment — Title + Body (Full Input)

Runs FinBERT on the full text input for each article. The input is constructed 
as follows:

- **Articles with valid body (`body_valid = True`):** title concatenated with 
  the first 1,000 characters of the body text. These 7,756 articles provide 
  FinBERT with enough context to produce a well-informed sentiment classification.

- **Articles without valid body (`body_valid = False`):** title only, used as 
  fallback. These 5,934 articles failed body scraping, were behind a paywall, 
  or had body text that did not pass the quality filter. Using the title as 
  fallback ensures every article receives a sentiment score rather than being 
  discarded entirely.

The resulting `full_sentiment` column is the primary sentiment variable used 
in all downstream modeling. It represents the best available sentiment signal 
for each article given the data constraints of web scraping at scale.

In [24]:
print("Running FinBERT on body text...")

def prepare_body(row):
    title = str(row['title']) if pd.notna(row['title']) else ''
    if row.get('body_valid') == True:
        return title + '. ' + str(row['body'])[:1000]
    return title

texts_full = df.apply(prepare_body, axis=1).tolist()
sentiment_full = get_sentiment(texts_full)
sentiment_full.columns = [f"full_{c}" for c in sentiment_full.columns]
print(sentiment_full['full_sentiment'].value_counts())

Running FinBERT on body text...
Processed 800/13690
Processed 1600/13690
Processed 2400/13690
Processed 3200/13690
Processed 4000/13690
Processed 4800/13690
Processed 5600/13690
Processed 6400/13690
Processed 7200/13690
Processed 8000/13690
Processed 8800/13690
Processed 9600/13690
Processed 10400/13690
Processed 11200/13690
Processed 12000/13690
Processed 12800/13690
Processed 13600/13690
full_sentiment
negative    6144
positive    4160
neutral     3386
Name: count, dtype: int64


### Merge and save

Save both results from headers and headers + body.

In [25]:
df_sentiment = pd.concat([df, sentiment_titles, sentiment_full], axis=1)

print(f"\n=== Headline Bias Analysis ===")
divergence = (df_sentiment['title_sentiment'] != df_sentiment['full_sentiment']).sum()
print(f"Divergent sentiment (title vs full): {divergence} ({divergence/len(df_sentiment)*100:.1f}%)")

df_sentiment.to_csv("../01_data/features/gdelt_wti_sentiment.csv", index=False)
print("Saved to 01_data/features/gdelt_wti_sentiment.csv")


=== Headline Bias Analysis ===
Divergent sentiment (title vs full): 3228 (23.6%)
Saved to 01_data/features/gdelt_wti_sentiment.csv


### Analyze divergence cases

Here we analyze when a news diverged its score from header vs header + body.

In [26]:
# Diagnostic — review divergent cases between title-only and title+body sentiment
divergence = (df_sentiment['title_sentiment'] != df_sentiment['full_sentiment']).sum()
total = len(df_sentiment)
print(f"=== Headline Bias Analysis ===")
print(f"Total articles: {total}")
print(f"Divergent sentiment (title vs full): {divergence} ({divergence/total*100:.1f}%)")
print(f"Agreement: {total - divergence} ({(total - divergence)/total*100:.1f}%)")

# Only show divergence for articles with valid body — title-only fallbacks will always agree
valid_body_mask = df_sentiment['body_valid'] == True
df_valid = df_sentiment[valid_body_mask]
divergence_valid = (df_valid['title_sentiment'] != df_valid['full_sentiment']).sum()
print(f"\nAmong articles with valid body ({len(df_valid)}):")
print(f"Divergent: {divergence_valid} ({divergence_valid/len(df_valid)*100:.1f}%)")
print(f"Agreement: {len(df_valid) - divergence_valid} ({(len(df_valid) - divergence_valid)/len(df_valid)*100:.1f}%)")

# Sample of divergent cases
print(f"\n--- Sample of 10 divergent cases ---")
df_div = df_valid[df_valid['title_sentiment'] != df_valid['full_sentiment']].head(10)

for _, row in df_div.iterrows():
    print(f"\nTitle: {row['title']}")
    print(f"Body preview: {str(row['body'])[:150]}")
    print(f"Title sentiment: {row['title_sentiment']} (conf: {row['title_confidence']})")
    print(f"Full sentiment:  {row['full_sentiment']} (conf: {row['full_confidence']})")
    print("-" * 60)

=== Headline Bias Analysis ===
Total articles: 13690
Divergent sentiment (title vs full): 3228 (23.6%)
Agreement: 10462 (76.4%)

Among articles with valid body (7755):
Divergent: 3228 (41.6%)
Agreement: 4527 (58.4%)

--- Sample of 10 divergent cases ---

Title: Oil Edges Up In Cautious Trade
Body preview: Oil prices were seeing modest gains in European trade on Monday after settling sharply lower last week on concerns over slowing demand in China, evide
Title sentiment: positive (conf: 0.7852)
Full sentiment:  neutral (conf: 0.6295)
------------------------------------------------------------

Title: Forex Today : Investor attention is expected to be on US CPI
Body preview: Following the release of February’s NFP, theUS Dollarregained some fresh upside traction and lifted the USD Index (DXY) from recent multi-week lows, w
Title sentiment: neutral (conf: 0.917)
Full sentiment:  positive (conf: 0.7012)
------------------------------------------------------------

Title: Asia markets live